# 260601 DL Training

| 항목 | 내용 |
|------|------|
| 데이터 | `outputs/260526_preprocessing/` |
| 예측인자 | VIF 생존 피처 (`vif_survived.csv`) |
| 결측 처리 | KNN / MICE / Mean / Median / Most-Frequent / Constant (노트북에서 선택) |
| CV | 5-Fold (평가용, 하이퍼파라미터 탐색 없음) |
| 모델 | PyTorch MLP (아키텍처 직접 설정) |
| 평가 지표 | Accuracy / Sensitivity / Specificity / Precision / F1 / AUC |
| 저장 | Confusion Matrix, ROC Curve, PR Curve, CSV |

---

## 레이어 구조

```
Input
  └─▶ [ Dense → BatchNorm → Activation → Dropout ] × N hidden layers
          └─▶ Dense(1)  ← 출력 logit
               └─▶ Sigmoid  ← 예측 확률
```

- **Loss**: `BCEWithLogitsLoss` (pos_weight로 클래스 불균형 자동 보정)
- **Optimizer**: Adam
- **Early stopping**: 검증 손실 미개선 시 조기 종료

---

## 5-Fold CV

```
전체 데이터
└── 5-Fold (StratifiedKFold)
     └── [Fold 1~5 반복]
          ├── Train (80%) → Imputer + StandardScaler fit → MLP fit
          └── Test  (20%) → transform → predict → 지표 계산
                              → mean ± std 보고
```

> Imputer / StandardScaler 모두 Train fit → Test transform (data leakage 방지)


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.dl_trainer import FeatureInspector, DLConfig, DLPipeline

## 공통 설정

아래 셀에서 **MLP 아키텍처**, **결측치 처리 전략**, **학습 파라미터**를 직접 수정한다.

In [2]:
# ── CV 설정 ───────────────────────────────────────────────────────
N_FOLDS      = 4
RANDOM_STATE = 42

# ── 결측치 처리 전략 ──────────────────────────────────────────────
# "knn"          : KNNImputer — k개 최근접 이웃 평균으로 대체
# "mice"         : IterativeImputer — MICE (Multiple Imputation by Chained Equations)
#                  피처 간 관계를 반복 회귀로 모델링하여 결측치를 예측 대체
# "mean"         : 각 피처의 평균으로 대체
# "median"       : 각 피처의 중앙값으로 대체
# "most_frequent": 각 피처의 최빈값으로 대체
# "constant"     : IMPUTATION_FILL_VALUE 값으로 일괄 대체
IMPUTATION_STRATEGY   = "knn"
KNN_N_NEIGHBORS       = 40
MICE_MAX_ITER         = 20    # MICE 전략 사용 시 최대 반복 횟수
IMPUTATION_FILL_VALUE = 0.0

# ── MLP 아키텍처 ──────────────────────────────────────────────────
# 각 은닉층은 Dense → BatchNorm → Activation → Dropout 순서로 구성된다.
#
# hidden_layer_sizes : 은닉층 뉴런 수 튜플
#   예) (64,)              → 은닉층 1개, 뉴런 64개
#       (128, 64)          → 은닉층 2개
#       (256, 128, 64)     → 은닉층 3개
#       (256, 128, 64, 32) → 은닉층 4개
HIDDEN_LAYER_SIZES = (64, 128, 256)

# dropout_rate : Dropout 비율 (0.0 이면 비활성화)
DROPOUT_RATE = 0.3

# ── 손실 함수 ─────────────────────────────────────────────────────
# "bce"          : 일반 BCEWithLogitsLoss
# "weighted_bce" : pos_weight = neg_count / pos_count
#                  소수 클래스(positive)에 더 높은 가중치를 자동 부여
LOSS = "weighted_bce"

# ── 최적화 ────────────────────────────────────────────────────────
LEARNING_RATE = 1e-3   # Adam 학습률
WEIGHT_DECAY  = 1e-4   # Adam L2 패널티 (가중치 감쇠)
BATCH_SIZE    = 16     # 미니배치 크기

# ── 학습 종료 조건 ────────────────────────────────────────────────
MAX_EPOCHS          = 1000   # 최대 에폭 수
EARLY_STOPPING      = True  # True이면 검증 손실 미개선 시 조기 종료
PATIENCE            = 40    # 조기 종료 전 개선 없는 최대 에폭 수
VALIDATION_FRACTION = 0.2   # early_stopping 검증셋 비율

# ── 예측 인자 직접 지정 (None 이면 vif_survived.csv 자동 로드) ────
DIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'Ca', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'K', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Na', 'Neutroph', 'P', 'PCT', 'PDW', 'PH', 'PSA', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '철포화율', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']
PREDIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'CRP', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Neutroph', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']


# ─────────────────────────────────────────────────────────────────

_ = FeatureInspector.show(ROOT / "outputs" / "260526_preprocessing" / "diabetes_dataset.xlsx",     title="Diabetes")
_ = FeatureInspector.show(ROOT / "outputs" / "260526_preprocessing" / "pre_diabetes_dataset.xlsx", title="Pre-diabetes")

── Diabetes (87개) ──
['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'Ca', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'K', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Na', 'Neutroph', 'P', 'PCT', 'PDW', 'PH', 'PSA', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '철포화율', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

── Pre-diabetes (82개) ──
['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BM

---
## 1. Diabetes (당뇨병전단계 → 당뇨병)

In [ ]:
diabetes_config = DLConfig(
    dataset_path          = ROOT / "outputs" / "260526_preprocessing" / "diabetes_dataset.xlsx",
    vif_features_path     = ROOT / "outputs" / "260527_statistic_analysis" / "diabetes" / "vif_survived.csv",
    output_dir            = ROOT / "outputs" / "260601_DL_training" / "diabetes",
    label_col             = "label",
    n_outer_folds         = N_FOLDS,
    random_state          = RANDOM_STATE,
    imputation_strategy   = IMPUTATION_STRATEGY,
    knn_n_neighbors       = KNN_N_NEIGHBORS,
    mice_max_iter         = MICE_MAX_ITER,
    imputation_fill_value = IMPUTATION_FILL_VALUE,
    hidden_layer_sizes    = HIDDEN_LAYER_SIZES,
    dropout_rate          = DROPOUT_RATE,
    loss                  = LOSS,
    learning_rate         = LEARNING_RATE,
    weight_decay          = WEIGHT_DECAY,
    batch_size            = BATCH_SIZE,
    max_epochs            = MAX_EPOCHS,
    early_stopping        = EARLY_STOPPING,
    patience              = PATIENCE,
    validation_fraction   = VALIDATION_FRACTION,
    selected_features     = DIABETES_FEATURES,
)

diabetes_pipeline = DLPipeline(diabetes_config)
diabetes_result   = diabetes_pipeline.run()

DL Pipeline    : diabetes_dataset.xlsx
Output dir     : /data2/mason/prediabetes_diabetes/outputs/260601_DL_training/diabetes
Device         : CPU
Imputation     : KNN (k=40)
Architecture   : (64, 128, 256)  (3 hidden layers)
Block          : Dense → BatchNorm → ReLU → Dropout(0.3)
Loss           : weighted_bce
Optimizer      : Adam  lr=0.001  weight_decay=0.0001
Batch size     : 16
Max epochs     : 1000  early_stopping=True  patience=40


/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


피처 소스: 직접 지정 (87개)
X shape: (629, 87), y distribution: {0: 600, 1: 29}
선택된 피처 (87개): ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'Ca', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'K', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Na', 'Neutroph', 'P', 'PCT', 'PDW', 'PH', 'PSA', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '철포화율', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

================

### 1-1. Fold별 상세 지표

In [ ]:
from IPython.display import display

display(diabetes_result.metrics_df().set_index("fold"))

,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc,threshold,n_epochs
fold,,,,,,,,,
1,0.987342,0.875000,0.993333,0.875000,0.875000,0.992500,0.847768,0.825673,55
2,0.980892,0.714286,0.993333,0.833333,0.769231,0.993333,0.754592,0.934633,48
3,0.955414,0.000000,1.000000,0.000000,0.000000,0.972381,0.658617,0.987250,53
4,0.961783,0.571429,0.980000,0.571429,0.571429,0.982857,0.716492,0.927431,50


### 1-2. 전체 요약 (mean ± std)

In [ ]:
diabetes_pipeline.exporter.print_summary_table(diabetes_result)

,mean,std,mean±std
metric,,,
accuracy,0.971358,0.015191,0.9714 ± 0.0152
sensitivity,0.540179,0.380871,0.5402 ± 0.3809
specificity,0.991667,0.008389,0.9917 ± 0.0084
precision,0.569940,0.403018,0.5699 ± 0.4030
f1,0.553915,0.390122,0.5539 ± 0.3901
auc,0.985268,0.009819,0.9853 ± 0.0098
pr_auc,0.744367,0.079428,0.7444 ± 0.0794
threshold,0.918747,0.067535,0.9187 ± 0.0675
n_epochs,51.500000,3.109126,51.5000 ± 3.1091


---
## 2. Pre-diabetes (정상 → 당뇨병전단계)

In [ ]:
prediabetes_config = DLConfig(
    dataset_path          = ROOT / "outputs" / "260526_preprocessing" / "pre_diabetes_dataset.xlsx",
    vif_features_path     = ROOT / "outputs" / "260527_statistic_analysis" / "pre_diabetes" / "vif_survived.csv",
    output_dir            = ROOT / "outputs" / "260601_DL_training" / "pre_diabetes",
    label_col             = "label",
    n_outer_folds         = N_FOLDS,
    random_state          = RANDOM_STATE,
    imputation_strategy   = IMPUTATION_STRATEGY,
    knn_n_neighbors       = KNN_N_NEIGHBORS,
    mice_max_iter         = MICE_MAX_ITER,
    imputation_fill_value = IMPUTATION_FILL_VALUE,
    hidden_layer_sizes    = HIDDEN_LAYER_SIZES,
    dropout_rate          = DROPOUT_RATE,
    loss                  = LOSS,
    learning_rate         = LEARNING_RATE,
    weight_decay          = WEIGHT_DECAY,
    batch_size            = BATCH_SIZE,
    max_epochs            = MAX_EPOCHS,
    early_stopping        = EARLY_STOPPING,
    patience              = PATIENCE,
    validation_fraction   = VALIDATION_FRACTION,
    selected_features     = PREDIABETES_FEATURES,
)

prediabetes_pipeline = DLPipeline(prediabetes_config)
prediabetes_result   = prediabetes_pipeline.run()

DL Pipeline    : pre_diabetes_dataset.xlsx
Output dir     : /data2/mason/prediabetes_diabetes/outputs/260601_DL_training/pre_diabetes
Device         : CPU
Imputation     : KNN (k=40)
Architecture   : (64, 128, 256)  (3 hidden layers)
Block          : Dense → BatchNorm → ReLU → Dropout(0.3)
Loss           : weighted_bce
Optimizer      : Adam  lr=0.001  weight_decay=0.0001
Batch size     : 16
Max epochs     : 1000  early_stopping=True  patience=40
피처 소스: 직접 지정 (82개)
X shape: (1240, 82), y distribution: {0: 1011, 1: 229}
선택된 피처 (82개): ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'CRP', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Neutroph', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride',

### 2-1. Fold별 상세 지표

In [ ]:
display(prediabetes_result.metrics_df().set_index("fold"))

,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc,threshold,n_epochs
fold,,,,,,,,,
1,0.909677,0.719298,0.952569,0.773585,0.745455,0.957978,0.856720,0.847533,51
2,0.948387,0.807018,0.980237,0.901961,0.851852,0.965051,0.920116,0.863898,67
3,0.948387,0.824561,0.976285,0.886792,0.854545,0.966368,0.933239,0.882279,56
4,0.941935,0.862069,0.960317,0.833333,0.847458,0.968596,0.889498,0.845227,50


### 2-2. 전체 요약 (mean ± std)

In [ ]:
prediabetes_pipeline.exporter.print_summary_table(prediabetes_result)

,mean,std,mean±std
metric,,,
accuracy,0.937097,0.018531,0.9371 ± 0.0185
sensitivity,0.803237,0.060487,0.8032 ± 0.0605
specificity,0.967352,0.013087,0.9674 ± 0.0131
precision,0.848918,0.058212,0.8489 ± 0.0582
f1,0.824827,0.052996,0.8248 ± 0.0530
auc,0.964498,0.004587,0.9645 ± 0.0046
pr_auc,0.899894,0.034122,0.8999 ± 0.0341
threshold,0.859734,0.017175,0.8597 ± 0.0172
n_epochs,56.000000,7.788881,56.0000 ± 7.7889
